# 🌊 Underwater Waste Detection System

This notebook trains a YOLOv8 model to detect and classify underwater waste objects.

## Features:
- ✅ Detects 15 types of ocean waste
- ✅ Environmental threat scoring (0-10 scale)
- ✅ Beautiful visualizations
- ✅ Batch testing capabilities
- ✅ Statistics and analytics

## Dataset:
- **Classes**: Mask, can, cellphone, electronics, gbottle, glove, metal, misc, net, pbag, pbottle, plastic, rod, sunglasses, tire
- **Total Images**: ~3000
- **Split**: train/valid/test

## 📦 Step 1: Setup and Installation

In [ ]:
# Install required packages
!pip install -q ultralytics opencv-python matplotlib seaborn pandas

# Import libraries
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
import pandas as pd
from google.colab import files
import zipfile
import shutil
import yaml
from ultralytics import YOLO

print("✅ Setup complete!")

## 📁 Step 2: Upload and Extract Dataset

In [ ]:
# Upload your dataset zip file
print("📁 Upload your dataset.zip file:")
uploaded = files.upload()

# Extract
zip_filename = list(uploaded.keys())[0]
print(f"\nExtracting {zip_filename}...")

with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall('dataset')

print("✅ Dataset extracted!")

## 🔍 Step 3: Verify Dataset Structure

In [ ]:
# Check dataset structure
train_path = Path('dataset/Underwater_garbage/train')
valid_path = Path('dataset/Underwater_garbage/valid')
test_path = Path('dataset/Underwater_garbage/test')

print("📊 Dataset Statistics:")
print("=" * 60)

if train_path.exists():
    train_img = list((train_path / 'images').glob('*'))
    train_lbl = list((train_path / 'labels').glob('*.txt'))
    print(f"✅ Train images: {len(train_img)}")
    print(f"✅ Train labels: {len(train_lbl)}")

if valid_path.exists():
    valid_img = list((valid_path / 'images').glob('*'))
    valid_lbl = list((valid_path / 'labels').glob('*.txt'))
    print(f"✅ Valid images: {len(valid_img)}")
    print(f"✅ Valid labels: {len(valid_lbl)}")

if test_path.exists():
    test_img = list((test_path / 'images').glob('*'))
    test_lbl = list((test_path / 'labels').glob('*.txt'))
    print(f"✅ Test images: {len(test_img)}")
    print(f"✅ Test labels: {len(test_lbl)}")

print("\n✅ Dataset verified!")

## 🎯 Step 4: Train YOLOv8 Model

In [ ]:
# Initialize YOLOv8 model
model = YOLO("yolov8n.pt")  # nano model (fast training)

# Training configuration
print("🚀 Starting training...")
print("Configuration:")
print("  - Model: YOLOv8n (nano)")
print("  - Epochs: 50")
print("  - Image size: 640")
print("  - Batch size: 8")
print("\nThis will take approximately 30-45 minutes...\n")

# Train the model
results = model.train(
    data="data.yaml",
    epochs=50,
    imgsz=640,
    batch=8,
    device=0,  # GPU
    workers=0,
    name='ocean_waste_detector',
    patience=15,
    save=True,
    plots=True
)

print("\n✅ Training complete!")
print(f"Best model saved at: runs/detect/ocean_waste_detector/weights/best.pt")

## 📊 Step 5: Load Trained Model and Setup Threat Scoring

In [ ]:
# Load the best trained model
model = YOLO('runs/detect/ocean_waste_detector/weights/best.pt')

# Class names
CLASS_NAMES = ['Mask', 'can', 'cellphone', 'electronics', 'gbottle', 'glove',
               'metal', 'misc', 'net', 'pbag', 'pbottle', 'plastic', 'rod',
               'sunglasses', 'tire']

# Environmental threat scores (0-10 scale)
THREAT_SCORES = {
    'net': 10,           # Critical - ghost fishing
    'pbag': 9,           # High - deadly to marine life
    'electronics': 9,    # High - toxic materials
    'cellphone': 9,      # High - batteries
    'pbottle': 8,        # High - persistent pollution
    'plastic': 8,        # High - microplastics
    'Mask': 8,           # High - COVID waste
    'tire': 8,           # High - toxic chemicals
    'metal': 7,          # Moderate-high
    'rod': 7,            # Moderate-high
    'glove': 7,          # Moderate-high
    'can': 6,            # Moderate
    'sunglasses': 6,     # Moderate
    'misc': 6,           # Moderate
    'gbottle': 5,        # Moderate-low
}

def get_threat_level(score):
    if score >= 9:
        return "CRITICAL"
    elif score >= 7:
        return "HIGH"
    elif score >= 5:
        return "MODERATE"
    else:
        return "LOW"

print("✅ Model loaded!")
print("\n📋 Threat Scoring System:")
print("=" * 60)
for cls in CLASS_NAMES:
    score = THREAT_SCORES.get(cls, 6)
    level = get_threat_level(score)
    print(f"{cls:<15} Score: {score}/10  Level: {level}")

## 🧪 Step 6: Test on Sample Images

In [ ]:
def analyze_image(image_path, confidence=0.25):
    """Analyze image and calculate threat scores"""
    results = model(str(image_path), conf=confidence)

    detections = []
    total_threat = 0

    for result in results:
        boxes = result.boxes
        for box in boxes:
            class_id = int(box.cls[0])
            class_name = CLASS_NAMES[class_id]
            confidence = float(box.conf[0])
            bbox = box.xyxy[0].cpu().numpy()

            threat_score = THREAT_SCORES.get(class_name, 6)
            threat_level = get_threat_level(threat_score)

            detections.append({
                'class_name': class_name,
                'confidence': confidence,
                'bbox': bbox,
                'threat_score': threat_score,
                'threat_level': threat_level
            })
            total_threat += threat_score

    avg_threat = total_threat / len(detections) if detections else 0
    overall_level = get_threat_level(avg_threat)

    return {
        'detections': detections,
        'total_objects': len(detections),
        'total_threat_score': total_threat,
        'avg_threat_score': avg_threat,
        'overall_threat_level': overall_level
    }

# Test on sample image
test_images = list(Path('dataset/Underwater_garbage/test/images').glob('*.jpg'))[:5]

if test_images:
    sample_img = test_images[0]
    print(f"Testing: {sample_img.name}\n")

    analysis = analyze_image(sample_img)

    print(f"📊 Results:")
    print(f"  Objects detected: {analysis['total_objects']}")
    print(f"  Average threat: {analysis['avg_threat_score']:.1f}/10")
    print(f"  Threat level: {analysis['overall_threat_level']}\n")

    if analysis['detections']:
        print(f"🔍 Detected objects:")
        for i, det in enumerate(analysis['detections'], 1):
            print(f"  {i}. {det['class_name']} - "
                  f"Conf: {det['confidence']:.1%} - "
                  f"Threat: {det['threat_score']}/10")

## 🎨 Step 7: Visualize Detections

In [ ]:
def visualize_detections(image_path, analysis, save_path='detection.jpg'):
    """Create visualization with threat-based colors"""

    image = cv2.imread(str(image_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    threat_colors = {
        'CRITICAL': (255, 0, 0),
        'HIGH': (255, 128, 0),
        'MODERATE': (255, 255, 0),
        'LOW': (128, 255, 0),
    }

    for det in analysis['detections']:
        bbox = det['bbox'].astype(int)
        x1, y1, x2, y2 = bbox
        color = threat_colors.get(det['threat_level'], (255, 255, 255))

        cv2.rectangle(image, (x1, y1), (x2, y2), color, 3)

        label = f"{det['class_name']} {det['confidence']:.2f}"
        threat_label = f"Threat: {det['threat_score']}/10"

        cv2.rectangle(image, (x1, y1 - 50), (x1 + 200, y1), color, -1)
        cv2.putText(image, label, (x1 + 5, y1 - 30),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)
        cv2.putText(image, threat_label, (x1 + 5, y1 - 10),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)

    plt.figure(figsize=(16, 12))
    plt.imshow(image)
    plt.axis('off')
    plt.title(f"Ocean Waste Detection - {analysis['overall_threat_level']} Threat",
             fontsize=18, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

# Visualize
if test_images:
    print("🎨 Creating visualization...")
    visualize_detections(sample_img, analysis)
    print("✅ Visualization complete!")

## 📈 Step 8: Generate Statistics

In [ ]:
# Batch analyze all test images
all_detections = []

print("Analyzing all test images...\n")
for img_path in test_images:
    analysis = analyze_image(img_path)
    all_detections.extend(analysis['detections'])

# Class distribution chart
class_counts = defaultdict(int)
for det in all_detections:
    class_counts[det['class_name']] += 1

sorted_classes = sorted(class_counts.items(), key=lambda x: x[1], reverse=True)
classes = [c[0] for c in sorted_classes]
counts = [c[1] for c in sorted_classes]

plt.figure(figsize=(14, 8))
plt.barh(classes, counts, color='#3498db', edgecolor='black', linewidth=1.5)
plt.xlabel('Number Detected', fontsize=13, fontweight='bold')
plt.ylabel('Waste Type', fontsize=13, fontweight='bold')
plt.title('Ocean Waste Distribution', fontsize=16, fontweight='bold')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('waste_distribution.jpg', dpi=150)
plt.show()

# Print statistics
print("\n📊 Overall Statistics:")
print("=" * 60)
print(f"Total objects detected: {len(all_detections)}")
print(f"\nTop 5 most common waste types:")
for cls, count in sorted_classes[:5]:
    score = THREAT_SCORES.get(cls, 6)
    print(f"  • {cls}: {count} (threat: {score}/10)")

avg_threat = sum(det['threat_score'] for det in all_detections) / len(all_detections)
print(f"\nAverage threat score: {avg_threat:.2f}/10")
print(f"Overall assessment: {get_threat_level(avg_threat)}")

## 💾 Step 9: Download Model and Results

In [ ]:
# Download trained model
print("Downloading trained model...")
files.download('runs/detect/ocean_waste_detector/weights/best.pt')

# Download visualizations
print("Downloading visualizations...")
files.download('waste_distribution.jpg')

print("\n✅ All files downloaded!")
print("\n🎉 Project complete!")

## 🚀 Next Steps

Your ocean waste detection system is complete! You can now:

1. **Use the trained model** on new underwater images
2. **Deploy the path optimizer** to calculate collection routes
3. **Integrate with underwater robots** for automated cleanup

### Path Optimization
Use the separate `ocean_waste_path_optimizer.py` script to:
- Detect trash in images
- Group nearby objects
- Calculate shortest collection path
- Get step-by-step route instructions

---

**Project**: Underwater Waste Detection & Collection  
**Model**: YOLOv8  
**Classes**: 15 ocean waste types  
**Features**: Detection, classification, threat scoring, path optimization